# Stage 2 — Adaptive Synthetic Allocation & DiffusionBoost

This notebook is the **single entry point** for Stage 2: migrations, CIFAR-100 SD generation (optional flag), **global FID** (clean-fid), Tiny ImageNet full grid, reduced CIFAR-100 track, and figures.

Training uses **synthetic-aware weighted CE** when `synthetic_aware_loss: true` in YAML and a same-architecture baseline checkpoint is passed (notebook does this automatically). Evaluation writes **temperature scaling**, **linear probe** (sklearn on frozen features), **covariance eigen-spectrum** (+ `eval_eigenvalues.png`), and **linear CKA vs baseline** (+ `eval_cka_heatmap.png`) into each run’s `metrics.json`.

**Prerequisites:** Stage 1 data (`data/tiny_imagenet_5pct/train_index.csv`, raw Tiny ImageNet). Synthetic pool: `data/synthetic_sd/` or `data/synthetic/tiny_imagenet/` (migration links legacy layout).

Use **Run All** after editing flags in the next cell (`RUN_CIFAR_GENERATE` / `RUN_FID` are heavy).

In [1]:
# ---- Run configuration ----
import os, sys, json
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)

# Master switches (edit before Run All) — all True = full end-to-end pipeline (long: CIFAR SD + FID + full Tiny grid)
RUN_MIGRATION = True          # symlink legacy synthetic_sd → canonical layout; link Stage 1 checkpoints
RUN_CIFAR_GENERATE = True     # generate CIFAR-100 synthetic cache if missing (very long SD run)
RUN_FID = True                # global FID Tiny: 5% real vs synthetic at each ratio (needs pip install clean-fid)
RUN_TINY_EXPERIMENTS = True   # full Tiny ImageNet training grid
RUN_CIFAR_EXPERIMENTS = True  # reduced CIFAR-100 (R18): baseline, uniform 15x, utility, adaptive, ceiling
RUN_FIGURES = True            # plot summary charts from results/ into figures/stage2/

# Tiny: set False to skip heavy sections during debugging
TINY_TRAIN_BASELINES = True
TINY_TRAIN_UNIFORM_SCALING = True   # ResNet-18 only: 5x, 10x, 15x
TINY_TRAIN_UNIFORM_15X_BOTH_ARCH = True
TINY_TRAIN_ADAPTIVE = True          # uses allocation CSVs (hard_class, uncertainty, predicted_utility)
TINY_TRAIN_CEILING = True

import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Project:", PROJECT_ROOT)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

Project: /home/bds/compvis
CUDA: True NVIDIA GeForce RTX 4060 Ti


## I. Environment & migrations
Resolves `data/synthetic_sd` into `data/synthetic/tiny_imagenet` (symlink or redirect file on Windows) and links `checkpoints/` under `results/tiny_imagenet/legacy/`.

In [2]:
from src.stage2.orchestrator import Stage2Orchestrator

orch = Stage2Orchestrator(PROJECT_ROOT)

if RUN_MIGRATION:
    synth_path = orch.migrate_tiny_synthetic()
    print("Synthetic (Tiny) resolved to:", synth_path)
    orch.link_stage1_checkpoints()
    print("Legacy checkpoints linked under results/.../legacy/")

if RUN_CIFAR_GENERATE:
    orch.ensure_cifar100_synthetic(force=False)
    print("CIFAR-100 synthetic ready under data/synthetic/cifar100")

if RUN_FID:
    fid_tiny = orch.compute_global_fid("tiny_imagenet.yaml", ratios=[5, 10, 15])
    print("Tiny ImageNet global FID:", fid_tiny)
    fid_cifar = orch.compute_global_fid("cifar100.yaml", ratios=[15])
    print("CIFAR-100 global FID (15x budget):", fid_cifar)

Synthetic (Tiny) resolved to: /home/bds/compvis/data/synthetic_sd
Legacy checkpoints linked under results/.../legacy/


/home/bds/compvis/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 169M/169M [00:14<00:00, 11.9MB/s] 


Loading runwayml/stable-diffusion-v1-5 ...


/home/bds/compvis/.venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading pipeline components...: 100%|██████████| 6/6 [00:26<00:00,  4.36s/it]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggin


CIFAR-100 generation complete in 1597.2 min
  Output: /home/bds/compvis/data/synthetic/cifar100
CIFAR-100 synthetic ready under data/synthetic/cifar100
compute FID between two folders
Found 5000 images in the folder /home/bds/compvis/results/tiny_imagenet/fid_cache/real_5pct_png


...

compute FID between two folders
Found 5000 images in the folder /home/bds/compvis/results/tiny_imagenet/fid_cache/real_5pct_png


FID real_5pct_png : 100%|██████████| 157/157 [00:47<00:00,  3.29it/s]


Found 50000 images in the folder /home/bds/compvis/results/tiny_imagenet/fid_cache/synth_uniform_10x_250pc


FID synth_uniform_10x_250pc : 100%|██████████| 1563/1563 [13:22<00:00,  1.95it/s]


OSError: [Errno 28] No space left on device: '/home/bds/compvis/data/synthetic_sd/n02480495/img_0006.png' -> '/home/bds/compvis/results/tiny_imagenet/fid_cache/synth_uniform_15x_375pc/n02480495_020447.png'

## II. Tiny ImageNet — training & evaluation
Order: baselines (R18 + MobileNetV3-Small) → uniform scaling (R18) → uniform 15× (both) → diagnostics & allocations → adaptive (per policy) → ceiling.
Metrics and checkpoints are written to `results/tiny_imagenet/{pipeline}/{arch}/{timestamp}/`.

In [ ]:
if not RUN_TINY_EXPERIMENTS:
    print("Skipping Tiny experiments.")
else:
    baseline_runs = {}
    uniform15 = {}

    if TINY_TRAIN_BASELINES:
        for arch in ["resnet18", "mobilenet_v3_small"]:
            rd, m = orch.train_baseline("tiny_imagenet.yaml", arch)
            baseline_runs[arch] = {"run_dir": str(rd), "metrics": m}
            print(arch, "baseline top1", m["top1"])

    r18_ckpt = Path(baseline_runs["resnet18"]["run_dir"]) / "best.pt" if baseline_runs.get("resnet18") else None

    def _tiny_ckpt(a):
        br = baseline_runs.get(a)
        return Path(br["run_dir"]) / "best.pt" if br else None

    if TINY_TRAIN_UNIFORM_SCALING and baseline_runs:
        ck = _tiny_ckpt("resnet18")
        for ratio in [5, 10, 15]:
            rd, m = orch.train_uniform(
                "tiny_imagenet.yaml", "resnet18", ratio, baseline_ckpt_same_arch=ck
            )
            print("uniform", ratio, "x top1", m["top1"])

    if TINY_TRAIN_UNIFORM_15X_BOTH_ARCH and baseline_runs:
        for arch in ["resnet18", "mobilenet_v3_small"]:
            ck = _tiny_ckpt(arch)
            rd, m = orch.train_uniform(
                "tiny_imagenet.yaml", arch, 15, baseline_ckpt_same_arch=ck
            )
            uniform15[arch] = {"run_dir": str(rd), "metrics": m}
            print(arch, "uniform 15x top1", m["top1"])

    utility_path = PROJECT_ROOT / "results" / "tiny_imagenet" / "utility_from_uniform15x.json"
    if uniform15.get("resnet18") and r18_ckpt and r18_ckpt.is_file():
        from src.data.registry import class_ids_in_label_order, get_baseline_loaders
        from src.data.transforms import get_train_transform, get_val_transform
        from src.config import load_experiment_config

        cfg = load_experiment_config(orch.config_path("tiny_imagenet.yaml"), PROJECT_ROOT)
        tr_t = get_train_transform(cfg.dataset.image_size)
        va_t = get_val_transform(cfg.dataset.image_size)
        _, _, c2i = get_baseline_loaders(cfg, tr_t, va_t)
        cids = class_ids_in_label_order(c2i)
        mb = baseline_runs["resnet18"]["metrics"]
        mu = uniform15["resnet18"]["metrics"]
        util = orch.utility_from_metrics(mb, mu, cids)
        utility_path.parent.mkdir(parents=True, exist_ok=True)
        utility_path.write_text(json.dumps(util, indent=2), encoding="utf-8")
        print("Saved utility targets to", utility_path)

        diag_path = orch.compute_baseline_diagnostics(
            "tiny_imagenet.yaml", r18_ckpt, arch="resnet18", quality_csv=None
        )
        print("Diagnostics CSV:", diag_path)

        alloc_dir = orch.build_allocations(
            "tiny_imagenet.yaml",
            diag_path,
            utility_path if utility_path.exists() else None,
            policies=["hard_class", "uncertainty", "predicted_utility"],
        )
        print("Allocation CSVs:", alloc_dir)

    if TINY_TRAIN_ADAPTIVE and baseline_runs:
        alloc_root = PROJECT_ROOT / "results" / "tiny_imagenet" / "allocations"
        for pol in ["hard_class", "uncertainty", "predicted_utility"]:
            csvp = alloc_root / f"allocation_{pol}.csv"
            if csvp.is_file():
                for arch in ["resnet18", "mobilenet_v3_small"]:
                    ck = _tiny_ckpt(arch)
                    rd, m = orch.train_adaptive(
                        "tiny_imagenet.yaml",
                        arch,
                        csvp,
                        name=f"adaptive_15x_{pol}",
                        baseline_ckpt_same_arch=ck,
                    )
                    print(arch, pol, "top1", m["top1"])

    if TINY_TRAIN_CEILING and baseline_runs:
        for arch in ["resnet18", "mobilenet_v3_small"]:
            ck = _tiny_ckpt(arch)
            rd, m = orch.train_ceiling(
                "tiny_imagenet.yaml", arch, baseline_ckpt_same_arch=ck
            )
            print(arch, "ceiling top1", m["top1"])

    orch.aggregate_results_index("tiny_imagenet.yaml")
    print("Tiny ImageNet results index updated.")

## III. CIFAR-100 (reduced generalisation track)
ResNet-18 only: **baseline** → **uniform 15×** (synthetic-aware CE + eval vs baseline CKA) → **per-class utility** → allocations (**hard_class** + **predicted_utility**) → **adaptive 15×** using `scope.cifar_adaptive_policy` in `configs/cifar100.yaml` (default **predicted_utility**) → **ceiling**.

In [ ]:
if not RUN_CIFAR_EXPERIMENTS:
    print("Skipping CIFAR experiments.")
else:
    rdir, mb = orch.train_baseline("cifar100.yaml", "resnet18")
    ck = Path(rdir) / "best.pt"
    _, m_uni = orch.train_uniform(
        "cifar100.yaml", "resnet18", 15, baseline_ckpt_same_arch=ck
    )
    from src.data.registry import class_ids_in_label_order, get_baseline_loaders
    from src.data.transforms import get_train_transform, get_val_transform
    from src.config import load_experiment_config

    cfg_c = load_experiment_config(orch.config_path("cifar100.yaml"), PROJECT_ROOT)
    tr_t = get_train_transform(cfg_c.dataset.image_size)
    va_t = get_val_transform(cfg_c.dataset.image_size)
    _, _, c2i = get_baseline_loaders(cfg_c, tr_t, va_t)
    cids = class_ids_in_label_order(c2i)
    util_c = orch.utility_from_metrics(mb, m_uni, cids)
    utility_path_c = PROJECT_ROOT / "results" / "cifar100" / "utility_from_uniform15x.json"
    utility_path_c.parent.mkdir(parents=True, exist_ok=True)
    utility_path_c.write_text(json.dumps(util_c, indent=2), encoding="utf-8")
    print("Saved CIFAR utility targets to", utility_path_c)

    diag_c = orch.compute_baseline_diagnostics("cifar100.yaml", ck, arch="resnet18")
    orch.build_allocations(
        "cifar100.yaml",
        diag_c,
        utility_path_c,
        policies=["hard_class", "predicted_utility"],
    )
    pol = cfg_c.scope.cifar_adaptive_policy
    acsv = PROJECT_ROOT / "results" / "cifar100" / "allocations" / f"allocation_{pol}.csv"
    if acsv.is_file():
        orch.train_adaptive(
            "cifar100.yaml",
            "resnet18",
            acsv,
            name=f"adaptive_15x_{pol}",
            baseline_ckpt_same_arch=ck,
        )
    else:
        print("Missing allocation CSV:", acsv)
    orch.train_ceiling("cifar100.yaml", "resnet18", baseline_ckpt_same_arch=ck)
    orch.aggregate_results_index("cifar100.yaml")
    print("CIFAR-100 reduced track complete.")

## IV. Figures (from saved results)
Reads `results/*/results_index.json` and writes quick comparison plots to `figures/stage2/`.

In [ ]:
if RUN_FIGURES:
    import matplotlib.pyplot as plt
    cfg_t = orch.load_cfg("tiny_imagenet.yaml")
    cfg_t.path_figures.mkdir(parents=True, exist_ok=True)
    idx_path = cfg_t.path_results_root / "tiny_imagenet" / "results_index.json"
    if idx_path.is_file():
        rows = json.loads(idx_path.read_text(encoding="utf-8"))
        # Last run per pipeline/arch
        best = {}
        for r in rows:
            key = (r["pipeline"], r["arch"])
            best[key] = r
        names = [f"{a[0]}\n{a[1]}" for a in sorted(best.keys())][:12]
        vals = [best[k]["top1"] for k in sorted(best.keys())][:12]
        plt.figure(figsize=(10, 4))
        plt.bar(range(len(names)), vals, color="steelblue")
        plt.xticks(range(len(names)), names, rotation=45, ha="right", fontsize=9)
        plt.ylabel("Val top-1")
        plt.tight_layout()
        outp = cfg_t.path_figures / "summary_top1_last_runs.png"
        plt.savefig(outp, dpi=300)
        plt.savefig(outp.with_suffix(".pdf"))
        plt.close()
        print("Saved", outp)
    else:
        print("No results index yet; run training cells first.")

## V. Submission checklist
- `results/` contains per-run `metrics.json`, `best.pt`, `training_curves.json`.
- Figures under `figures/stage2/` (PNG + PDF).
- Export Stage 2 report PDF with numbers filled from `metrics.json` / `results_index.json`.